In [1]:
# ── PDF & HTML → Markdown (base64 inline images) ─────────────────────────────
# Drop PDFs / HTML files in the same folder as this notebook and run.
# Output: files_as_md/<filename>.md  (one self-contained file per source)
# ─────────────────────────────────────────────────────────────────────────────
import subprocess, sys

def pip(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', pkg])

print('📦 Installing libraries...')
for pkg in ('pymupdf4llm', 'markdownify', 'beautifulsoup4', 'Pillow'):
    pip(pkg)
print('✅ Done.\n')

# ── imports ───────────────────────────────────────────────────────────────────
import re, base64, tempfile
from pathlib import Path
import pymupdf4llm
from markdownify import markdownify
from bs4 import BeautifulSoup

OUTPUT_DIR = Path('files_as_md')
OUTPUT_DIR.mkdir(exist_ok=True)

def to_b64_md(img_path: Path) -> str:
    """Return a markdown image tag with the image base64-encoded inline."""
    ext  = img_path.suffix.lstrip('.').lower()
    mime = 'jpeg' if ext in ('jpg', 'jpeg') else ext
    data = base64.b64encode(img_path.read_bytes()).decode()
    return f'![](data:image/{mime};base64,{data})'

# ── PDF converter ─────────────────────────────────────────────────────────────
def convert_pdf(src: Path) -> Path:
    with tempfile.TemporaryDirectory() as tmp:
        tmp_path = Path(tmp)
        md_text  = pymupdf4llm.to_markdown(
            str(src),
            write_images=True,
            image_path=str(tmp_path),
            image_format='png',
        )
        # Replace every file-based image reference with inline base64
        def inline(match):
            raw = match.group(1)
            img = Path(raw) if Path(raw).is_absolute() else tmp_path / Path(raw).name
            return to_b64_md(img) if img.exists() else match.group(0)

        md_text = re.sub(r'!\[.*?\]\((.+?)\)', inline, md_text)

    out = OUTPUT_DIR / f'{src.stem}.md'
    out.write_text(md_text, encoding='utf-8')
    return out

# ── HTML converter ────────────────────────────────────────────────────────────
def convert_html(src: Path) -> Path:
    html = src.read_text(encoding='utf-8', errors='replace')
    soup = BeautifulSoup(html, 'html.parser')

    for tag in soup(['script', 'style', 'noscript', 'iframe']):
        tag.decompose()

    # Inline every local image as base64 directly in the HTML before converting
    for img_tag in soup.find_all('img'):
        img_src = img_tag.get('src', '')
        if img_src.startswith('data:'):
            continue
        img_path = (src.parent / img_src).resolve()
        if img_path.exists():
            ext  = img_path.suffix.lstrip('.').lower()
            mime = 'jpeg' if ext in ('jpg', 'jpeg') else ext
            data = base64.b64encode(img_path.read_bytes()).decode()
            img_tag['src'] = f'data:image/{mime};base64,{data}'

    md_text = markdownify(str(soup), heading_style='ATX', bullets='-')
    md_text = re.sub(r'\n{3,}', '\n\n', md_text).strip()

    out = OUTPUT_DIR / f'{src.stem}.md'
    out.write_text(md_text, encoding='utf-8')
    return out

# ── scan & convert ────────────────────────────────────────────────────────────
HERE  = Path('.').resolve()
files = [
    f for f in HERE.iterdir()
    if f.is_file()
    and f.suffix.lower() in {'.pdf', '.html', '.htm'}
    and OUTPUT_DIR.resolve() not in f.parents
]

if not files:
    print('⚠️  No PDF or HTML files found in:', HERE)
else:
    print(f'🔍 Found {len(files)} file(s):\n')
    for src in files:
        print(f'  {src.name}  →  ', end='', flush=True)
        try:
            out      = convert_pdf(src) if src.suffix.lower() == '.pdf' else convert_html(src)
            size_kb  = round(out.stat().st_size / 1024)
            print(f'✅  {out.name}  ({size_kb} KB)')
        except Exception as e:
            print(f'❌  {e}')
    print(f'\n🎉 All done → {OUTPUT_DIR}/')

📦 Installing libraries...
✅ Done.

Consider using the pymupdf_layout package for a greatly improved page layout analysis.
🔍 Found 2 file(s):

  prof_guide_professor_notes.pdf  →  ✅  prof_guide_professor_notes.md  (424 KB)
  proj_guide_NLPS_-_TREC_2025_BioGen.pdf  →  ✅  proj_guide_NLPS_-_TREC_2025_BioGen.md  (2166 KB)

🎉 All done → files_as_md/
